# Ensemble & Deep Forest Experiments
## Πρωτεομική Ταξινόμηση Ογκολογικών Ασθενών

**Μοντέλα:**
1. Logistic Regression (baseline)
2. Random Forest (bagging)
3. Random Subspace (feature bagging)
4. Rotation Forest (PCA + trees)
5. Deep Forest - DF21 (pip package)
6. Deep Forest - Custom Cascade (sklearn)

**Σύγκριση με:** Vanilla RNN (~51%), LSTM (~35%)

In [ ]:
# === Εγκατάσταση deep-forest ===
!pip install deep-forest -q

In [ ]:
# === Imports ===
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from deepforest import CascadeForestClassifier

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)
print('Setup complete ✓')

In [ ]:
# === Configuration ===
class Config:
    RANDOM_SEED = 42
    N_FOLDS = 5
    N_TOP_FEATURES = 15          # <-- 15 πρωτεΐνες αντί 50
    
    # Random Forest
    RF_N_ESTIMATORS = 300
    RF_MAX_DEPTH = None          # Αφήνουμε τα δέντρα να μεγαλώσουν
    
    # Random Subspace
    RS_N_ESTIMATORS = 200
    RS_MAX_FEATURES = 0.5        # Κάθε δέντρο βλέπει 50% features
    
    # Rotation Forest
    ROT_N_ESTIMATORS = 100
    ROT_N_GROUPS = 3             # Χωρίζουμε 15 features σε 3 ομάδες × 5
    
    # Deep Forest
    DF_N_ESTIMATORS = 2          # Forests per cascade layer
    DF_N_TREES = 100             # Trees per forest
    
print('Config loaded ✓')

In [ ]:
# ============================================================
# ΑΛΛΑΞΕ ΑΥΤΑ ΤΑ PATHS ΣΤΑ ΔΙΚΑ ΣΟΥ ΑΡΧΕΙΑ
# ============================================================
SAMPLES_PATH = '/content/DIOPTRA_Samples_list_Oncology.xlsx'
PROTEINS_PATH = '/content/DIOPTRA_Protein_results_Oncology.xlsx'
# ============================================================

In [ ]:
# === Φόρτωση Δεδομένων (ίδιο pipeline με τα RNN notebooks) ===

def load_data(samples_path, proteins_path, include_t00=True):
    samples = pd.read_excel(samples_path)
    proteins = pd.read_excel(proteins_path)
    data = samples.merge(proteins, on='SampleID')
    protein_cols = [c for c in proteins.columns if c != 'SampleID']

    # Paired samples (T0 → T1)
    t0_data = data[data['Timepoint'] == 'T0'].copy()
    t1_data = data[data['Timepoint'] == 'T1'].copy()
    t0_data['PatientID'] = t0_data['SampleID']
    t1_data['PatientID'] = t1_data['SampleID'].str.replace('-F', '', regex=False)

    paired = t0_data.merge(
        t1_data[['PatientID'] + protein_cols],
        on='PatientID', suffixes=('_T0', '_T1')
    )
    print(f'Paired samples: {len(paired)}')

    if include_t00:
        t00_data = data[data['Timepoint'] == 'T00'].copy()
        for col in protein_cols:
            t00_data[f'{col}_T0'] = t00_data[col]
            t00_data[f'{col}_T1'] = t00_data[col]
        t00_data['PatientID'] = t00_data['SampleID']
        cols_needed = ['PatientID', 'Group'] + [f'{c}_T0' for c in protein_cols] + [f'{c}_T1' for c in protein_cols]
        combined = pd.concat([paired[cols_needed], t00_data[cols_needed]], ignore_index=True)
        print(f'After adding T00: {len(combined)}')
    else:
        combined = paired

    combined = combined[combined['Group'] != 'NAA']
    print(f'After removing NAA: {len(combined)}')

    X_T0 = combined[[f'{c}_T0' for c in protein_cols]].values
    X_T1 = combined[[f'{c}_T1' for c in protein_cols]].values
    X = np.stack([X_T0, X_T1], axis=1)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(combined['Group'].values)

    print(f'\nClass distribution:')
    for i, cls in enumerate(label_encoder.classes_):
        print(f'  {cls}: {(y == i).sum()}')

    return X, y, protein_cols, label_encoder

X_raw, y, protein_names, label_encoder = load_data(SAMPLES_PATH, PROTEINS_PATH)
class_names = list(label_encoder.classes_)
print(f'\nX shape: {X_raw.shape}  |  Classes: {class_names}')

In [ ]:
# === Προεπεξεργασία: NaN handling + Feature Selection (15 proteins) ===

# Βήμα 1: Αφαίρεση πρωτεϊνών με μόνο NaN
n_samples, n_timepoints, n_proteins = X_raw.shape
valid_mask = ~np.all(np.isnan(X_raw), axis=(0, 1))
X_raw = X_raw[:, :, valid_mask]
protein_names = [p for p, v in zip(protein_names, valid_mask) if v]
print(f'Valid proteins: {X_raw.shape[2]}')

# Βήμα 2: Median Imputation
imputer = SimpleImputer(strategy='median')
X_imp = np.zeros_like(X_raw)
for t in range(n_timepoints):
    X_imp[:, t, :] = imputer.fit_transform(X_raw[:, t, :])
X_raw = X_imp
print(f'NaN after imputation: {np.isnan(X_raw).sum()}')

# Βήμα 3: Feature Selection - Top 15 πρωτεΐνες (ANOVA F-score)
selector = SelectKBest(f_classif, k=Config.N_TOP_FEATURES)
selector.fit(X_raw[:, 0, :], y)  # Χρήση T0 για selection
selected_mask = selector.get_support()
selected_idx = np.where(selected_mask)[0]

X_selected = X_raw[:, :, selected_idx]
selected_proteins = [protein_names[i] for i in selected_idx]

print(f'\nSelected {len(selected_proteins)} proteins:')
# Εμφάνιση F-scores
f_scores = selector.scores_[selected_idx]
for rank, (p, f) in enumerate(sorted(zip(selected_proteins, f_scores), key=lambda x: -x[1]), 1):
    print(f'  {rank:2d}. {p:<12s} F={f:.2f}')

print(f'\nX_selected shape: {X_selected.shape}')

In [ ]:
# === Κατασκευή Feature Matrices για κάθε Mode ===
# Τα tree-based models δεν δέχονται 3D tensors, χρειάζονται 2D

X_T0 = X_selected[:, 0, :]                           # (N, 15) - Static
X_T1 = X_selected[:, 1, :]                           # (N, 15)
X_delta = X_T1 - X_T0                                # (N, 15) - Delta
X_temporal = np.hstack([X_T0, X_T1])                 # (N, 30) - Temporal (T0 + T1 concat)
X_full = np.hstack([X_T0, X_T1, X_delta])            # (N, 45) - Full (T0 + T1 + Delta)

print('Feature matrices:')
print(f'  X_T0 (Static):    {X_T0.shape}')
print(f'  X_delta (Delta):  {X_delta.shape}')
print(f'  X_temporal (T0+T1): {X_temporal.shape}')
print(f'  X_full (T0+T1+Δ): {X_full.shape}')

---
## Βοηθητικές Συναρτήσεις

In [ ]:
# === Evaluation Framework ===

def evaluate_model(model_fn, X, y, n_folds=5, model_name='Model', normalize=True):
    """
    Αξιολόγηση μοντέλου με Stratified K-Fold CV.
    
    Args:
        model_fn: callable που επιστρέφει ένα νέο (unfitted) μοντέλο
        X: feature matrix (2D)
        y: labels
        normalize: αν True, εφαρμόζει StandardScaler ανά fold
    
    Returns:
        dict με metrics
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=Config.RANDOM_SEED)
    
    fold_acc = []
    fold_f1 = []
    fold_precision = []
    fold_recall = []
    all_y_true = []
    all_y_pred = []
    feature_importances = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Normalization ανά fold (αποφυγή data leakage)
        if normalize:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)
        
        # Train & Predict
        model = model_fn()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='weighted', zero_division=0)
        prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_val, y_pred, average='weighted', zero_division=0)
        
        fold_acc.append(acc)
        fold_f1.append(f1)
        fold_precision.append(prec)
        fold_recall.append(rec)
        all_y_true.extend(y_val)
        all_y_pred.extend(y_pred)
        
        # Feature importance (αν υπάρχει)
        if hasattr(model, 'feature_importances_'):
            feature_importances.append(model.feature_importances_)
        
        print(f'  Fold {fold}: Acc={acc:.3f}  F1={f1:.3f}')
    
    results = {
        'model': model_name,
        'mean_acc': np.mean(fold_acc),
        'std_acc': np.std(fold_acc),
        'mean_f1': np.mean(fold_f1),
        'std_f1': np.std(fold_f1),
        'mean_precision': np.mean(fold_precision),
        'mean_recall': np.mean(fold_recall),
        'fold_accs': fold_acc,
        'y_true': all_y_true,
        'y_pred': all_y_pred,
        'feature_importances': np.mean(feature_importances, axis=0) if feature_importances else None
    }
    
    print(f'  >>> {model_name}: Acc={results["mean_acc"]:.3f}±{results["std_acc"]:.3f}  '
          f'F1={results["mean_f1"]:.3f}±{np.std(fold_f1):.3f}')
    return results

print('Evaluation framework ready ✓')

In [ ]:
# === Rotation Forest Implementation ===

class RotationForestClassifier:
    """
    Rotation Forest (Rodriguez et al., 2006)
    
    1. Χωρίζει τα features σε τυχαίες ομάδες
    2. Εφαρμόζει PCA σε κάθε ομάδα (σε bootstrap sample)
    3. Συνενώνει τα PCA components
    4. Εκπαιδεύει ένα Decision Tree στα μετασχηματισμένα features
    5. Επαναλαμβάνει Β φορές και κάνει majority voting
    """
    def __init__(self, n_estimators=100, n_groups=3, random_state=42):
        self.n_estimators = n_estimators
        self.n_groups = n_groups
        self.random_state = random_state
        self.trees = []
        self.rotations = []  # PCA objects per tree
        self.feature_groups = []  # Feature groupings per tree
    
    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n_features = X.shape[1]
        self.trees = []
        self.rotations = []
        self.feature_groups = []
        
        for i in range(self.n_estimators):
            # 1. Τυχαίος χωρισμός features σε ομάδες
            perm = rng.permutation(n_features)
            groups = np.array_split(perm, self.n_groups)
            
            # 2. PCA σε κάθε ομάδα (σε bootstrap subset)
            pca_list = []
            for group in groups:
                # Bootstrap sample (75% των δεδομένων)
                boot_idx = rng.choice(len(X), size=int(0.75 * len(X)), replace=True)
                X_subset = X[boot_idx][:, group]
                
                pca = PCA(n_components=len(group))
                pca.fit(X_subset)
                pca_list.append(pca)
            
            # 3. Μετασχηματισμός ΟΛΩΝ των δεδομένων
            X_rotated = self._transform(X, groups, pca_list)
            
            # 4. Εκπαίδευση Decision Tree
            tree = DecisionTreeClassifier(random_state=self.random_state + i)
            tree.fit(X_rotated, y)
            
            self.trees.append(tree)
            self.rotations.append(pca_list)
            self.feature_groups.append(groups)
        
        return self
    
    def _transform(self, X, groups, pca_list):
        """Εφαρμόζει PCA rotation σε κάθε ομάδα και συνενώνει"""
        parts = []
        for group, pca in zip(groups, pca_list):
            parts.append(pca.transform(X[:, group]))
        return np.hstack(parts)
    
    def predict(self, X):
        """Majority voting"""
        predictions = np.array([
            tree.predict(self._transform(X, groups, pcas))
            for tree, groups, pcas in zip(self.trees, self.feature_groups, self.rotations)
        ])
        # Majority vote ανά δείγμα
        from scipy.stats import mode
        return mode(predictions, axis=0, keepdims=False).mode

print('Rotation Forest ready ✓')

In [ ]:
# === Custom Cascade Forest Implementation ===

class CustomCascadeForest:
    """
    Custom υλοποίηση Cascade Forest (Zhou & Feng, 2017)
    χρησιμοποιώντας μόνο sklearn.
    
    Κάθε layer: [RF, RF, ExtraTrees, ExtraTrees]
    Κάθε forest δίνει class probabilities (μέσω 3-fold CV στο train set)
    Αυτές concatenate με τα αρχικά features → input για το επόμενο layer.
    Σταματάει όταν δεν βελτιώνεται.
    """
    def __init__(self, n_trees=100, max_layers=10, random_state=42):
        self.n_trees = n_trees
        self.max_layers = max_layers
        self.random_state = random_state
        self.layers = []
    
    def _create_layer(self, seed):
        """4 forests ανά layer για diversity"""
        return [
            RandomForestClassifier(n_estimators=self.n_trees, random_state=seed),
            RandomForestClassifier(n_estimators=self.n_trees, random_state=seed+1),
            ExtraTreesClassifier(n_estimators=self.n_trees, random_state=seed+2),
            ExtraTreesClassifier(n_estimators=self.n_trees, random_state=seed+3),
        ]
    
    def _get_cv_probas(self, forests, X, y):
        """
        Παράγει class probabilities μέσω 3-fold CV
        για αποφυγή data leakage μεταξύ layers.
        """
        n_classes = len(np.unique(y))
        probas = np.zeros((len(X), n_classes * len(forests)))
        
        kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=self.random_state)
        
        for f_idx, forest in enumerate(forests):
            proba_col = np.zeros((len(X), n_classes))
            
            for tr_idx, va_idx in kf.split(X, y):
                clf = forest.__class__(
                    n_estimators=self.n_trees,
                    random_state=self.random_state + f_idx
                )
                clf.fit(X[tr_idx], y[tr_idx])
                proba_col[va_idx] = clf.predict_proba(X[va_idx])
            
            probas[:, f_idx*n_classes:(f_idx+1)*n_classes] = proba_col
        
        return probas
    
    def fit(self, X, y):
        self.layers = []
        self.original_dim = X.shape[1]
        self.n_classes_ = len(np.unique(y))
        
        X_current = X.copy()
        best_acc = 0
        patience_counter = 0
        
        for layer_idx in range(self.max_layers):
            forests = self._create_layer(self.random_state + layer_idx * 10)
            
            # CV probabilities
            probas = self._get_cv_probas(forests, X_current, y)
            
            # Augmented features = original + probas
            X_augmented = np.hstack([X, probas])  # ΠΑΝΤΑ original X + new probas
            
            # Αξιολόγηση: μέσος όρος probabilities → accuracy
            n_forests = len(forests)
            avg_proba = np.zeros((len(X), self.n_classes_))
            for f_idx in range(n_forests):
                avg_proba += probas[:, f_idx*self.n_classes_:(f_idx+1)*self.n_classes_]
            avg_proba /= n_forests
            layer_acc = accuracy_score(y, np.argmax(avg_proba, axis=1))
            
            print(f'    Layer {layer_idx+1}: Acc={layer_acc:.3f}')
            
            # Fit the actual forests (on full data, for later prediction)
            for forest in forests:
                forest.fit(X_current, y)
            
            self.layers.append((forests, X_current.shape[1]))
            
            # Early stopping
            if layer_acc > best_acc + 0.001:
                best_acc = layer_acc
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 2:
                    print(f'    Early stop at layer {layer_idx+1}')
                    break
            
            X_current = X_augmented
        
        return self
    
    def predict(self, X):
        X_current = X.copy()
        
        for forests, expected_dim in self.layers:
            # Get probabilities from each forest
            probas_list = []
            for forest in forests:
                # Αν X_current δεν ταιριάζει, χρησιμοποιούμε original dims
                if X_current.shape[1] != expected_dim:
                    X_input = X_current[:, :expected_dim]
                else:
                    X_input = X_current
                probas_list.append(forest.predict_proba(X_input))
            
            # Augment
            all_probas = np.hstack(probas_list)
            X_current = np.hstack([X, all_probas])
        
        # Final prediction: average of last layer probabilities
        avg_proba = np.mean(probas_list, axis=0)
        return np.argmax(avg_proba, axis=1)

print('Custom Cascade Forest ready ✓')

---
## Εκτέλεση Πειραμάτων

Χρησιμοποιούμε **X_temporal (T0+T1)** ως κύριο input (30 features) 
αλλά δοκιμάζουμε και X_T0 (Static) για σύγκριση.

In [ ]:
# Αποθήκευση αποτελεσμάτων
all_results = []

# Ποιο input θα χρησιμοποιήσουμε κυρίως
X_main = X_temporal  # T0 + T1 concatenated
input_label = 'Temporal (T0+T1)'

In [ ]:
# === 1. Logistic Regression ===
print('='*60)
print('1. LOGISTIC REGRESSION')
print('='*60)

res = evaluate_model(
    model_fn=lambda: LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=Config.RANDOM_SEED,
        C=1.0,
        solver='lbfgs',
        multi_class='multinomial'
    ),
    X=X_main, y=y,
    model_name='Logistic Regression',
    normalize=True
)
all_results.append(res)

In [ ]:
# === 2. Random Forest ===
print('='*60)
print('2. RANDOM FOREST')
print('='*60)

res = evaluate_model(
    model_fn=lambda: RandomForestClassifier(
        n_estimators=Config.RF_N_ESTIMATORS,
        max_depth=Config.RF_MAX_DEPTH,
        class_weight='balanced',
        random_state=Config.RANDOM_SEED,
        n_jobs=-1
    ),
    X=X_main, y=y,
    model_name='Random Forest',
    normalize=False  # RF δεν χρειάζεται normalization
)
all_results.append(res)

In [ ]:
# === 3. Random Subspace ===
print('='*60)
print('3. RANDOM SUBSPACE (Feature Bagging)')
print('='*60)

res = evaluate_model(
    model_fn=lambda: BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=Config.RS_N_ESTIMATORS,
        max_features=Config.RS_MAX_FEATURES,  # 50% features per tree
        bootstrap=True,                        # Bootstrap samples too
        bootstrap_features=True,               # Random feature subsets
        random_state=Config.RANDOM_SEED,
        n_jobs=-1
    ),
    X=X_main, y=y,
    model_name='Random Subspace',
    normalize=False
)
all_results.append(res)

In [ ]:
# === 4. Rotation Forest ===
print('='*60)
print('4. ROTATION FOREST')
print('='*60)

res = evaluate_model(
    model_fn=lambda: RotationForestClassifier(
        n_estimators=Config.ROT_N_ESTIMATORS,
        n_groups=Config.ROT_N_GROUPS,
        random_state=Config.RANDOM_SEED
    ),
    X=X_main, y=y,
    model_name='Rotation Forest',
    normalize=True  # PCA χρειάζεται normalized data
)
all_results.append(res)

In [ ]:
# === 5. Deep Forest - DF21 (Official Package) ===
print('='*60)
print('5. DEEP FOREST (DF21 Package)')
print('='*60)

res = evaluate_model(
    model_fn=lambda: CascadeForestClassifier(
        n_estimators=Config.DF_N_ESTIMATORS,
        n_trees=Config.DF_N_TREES,
        random_state=Config.RANDOM_SEED,
        verbose=0
    ),
    X=X_main, y=y,
    model_name='Deep Forest (DF21)',
    normalize=False
)
all_results.append(res)

In [ ]:
# === 6. Custom Cascade Forest ===
print('='*60)
print('6. CUSTOM CASCADE FOREST')
print('='*60)

res = evaluate_model(
    model_fn=lambda: CustomCascadeForest(
        n_trees=Config.DF_N_TREES,
        max_layers=10,
        random_state=Config.RANDOM_SEED
    ),
    X=X_main, y=y,
    model_name='Custom Cascade Forest',
    normalize=False
)
all_results.append(res)

---
## Συγκριτικά Αποτελέσματα

In [ ]:
# === Συγκριτικός Πίνακας ===

# Προσθήκη RNN/LSTM αποτελεσμάτων από τα προηγούμενα πειράματα
rnn_results = [
    {'model': 'Vanilla RNN (T0+T1)', 'mean_acc': 0.513, 'std_acc': 0.116,
     'mean_f1': np.nan, 'std_f1': np.nan, 'mean_precision': np.nan, 'mean_recall': np.nan},
    {'model': 'LSTM (T0+T1)', 'mean_acc': 0.336, 'std_acc': 0.081,
     'mean_f1': np.nan, 'std_f1': np.nan, 'mean_precision': np.nan, 'mean_recall': np.nan},
]

# Δημιουργία DataFrame
rows = []
for r in all_results:
    rows.append({
        'Model': r['model'],
        'Mean Acc': f"{r['mean_acc']:.3f}",
        'Std Acc': f"{r['std_acc']:.3f}",
        'Mean F1': f"{r['mean_f1']:.3f}",
        'Mean Prec': f"{r['mean_precision']:.3f}",
        'Mean Rec': f"{r['mean_recall']:.3f}",
    })
for r in rnn_results:
    rows.append({
        'Model': r['model'],
        'Mean Acc': f"{r['mean_acc']:.3f}",
        'Std Acc': f"{r['std_acc']:.3f}",
        'Mean F1': '-',
        'Mean Prec': '-',
        'Mean Rec': '-',
    })

df_results = pd.DataFrame(rows)
print('\n' + '='*70)
print('ΣΥΓΚΡΙΤΙΚΟΣ ΠΙΝΑΚΑΣ ΑΠΟΤΕΛΕΣΜΑΤΩΝ')
print(f'Input: {input_label} | Features: {X_main.shape[1]} | N={len(y)} | 5-Fold CV')
print('='*70)
print(df_results.to_string(index=False))
print('='*70)

In [ ]:
# === Οπτικοποίηση: Bar Chart Σύγκρισης ===

fig, ax = plt.subplots(figsize=(12, 6))

# Ensemble results
models = [r['model'] for r in all_results]
accs = [r['mean_acc'] for r in all_results]
stds = [r['std_acc'] for r in all_results]

# Add RNN/LSTM
models += ['Vanilla RNN', 'LSTM']
accs += [0.513, 0.336]
stds += [0.116, 0.081]

# Colors: ensemble = teal, DL = coral
colors = ['#1D9E75'] * len(all_results) + ['#D85A30'] * 2

bars = ax.bar(range(len(models)), accs, yerr=stds, capsize=4,
              color=colors, edgecolor='white', linewidth=0.5, alpha=0.85)

# Baseline line
ax.axhline(y=1/3, color='gray', linestyle='--', alpha=0.5, label='Random (33%)')

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Mean Accuracy (5-Fold CV)', fontsize=12)
ax.set_title('Σύγκριση Μεθόδων: Ensemble vs Deep Learning\n'
             f'(15 πρωτεΐνες, N={len(y)}, 3 κλάσεις)', fontsize=13)
ax.set_ylim(0, 1.0)
ax.legend()

# Τιμές πάνω στα bars
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

In [ ]:
# === Confusion Matrices (για τα ensemble models) ===

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, r in enumerate(all_results):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[idx])
    axes[idx].set_title(f"{r['model']}\nAcc={r['mean_acc']:.3f}", fontsize=11)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

# Hide extra subplots
for idx in range(len(all_results), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Confusion Matrices - Ensemble Methods (5-Fold CV, Aggregated)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Feature Importance (Random Forest) ===

# Βρίσκουμε το RF result
rf_result = [r for r in all_results if r['model'] == 'Random Forest'][0]

if rf_result['feature_importances'] is not None:
    # Feature names for temporal input (T0 + T1)
    feat_names = ([f'{p}_T0' for p in selected_proteins] + 
                  [f'{p}_T1' for p in selected_proteins])
    
    importances = rf_result['feature_importances']
    sorted_idx = np.argsort(importances)[::-1][:20]  # Top 20
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(sorted_idx)), importances[sorted_idx][::-1],
            color='#1D9E75', alpha=0.8)
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feat_names[i] for i in sorted_idx][::-1], fontsize=10)
    ax.set_xlabel('Feature Importance (MDI)', fontsize=12)
    ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13)
    plt.tight_layout()
    plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: feature_importance_rf.png')

In [ ]:
# === Αποθήκευση Αποτελεσμάτων σε CSV ===

# Λεπτομερή αποτελέσματα
detailed_rows = []
for r in all_results:
    row = {
        'Model': r['model'],
        'Mean_Accuracy': r['mean_acc'],
        'Std_Accuracy': r['std_acc'],
        'Mean_F1': r['mean_f1'],
        'Mean_Precision': r['mean_precision'],
        'Mean_Recall': r['mean_recall'],
    }
    for i, acc in enumerate(r['fold_accs'], 1):
        row[f'Fold_{i}'] = acc
    detailed_rows.append(row)

df_detailed = pd.DataFrame(detailed_rows)
df_detailed.to_csv('ensemble_results.csv', index=False)
print('Saved: ensemble_results.csv')
print(df_detailed.to_string(index=False))

In [ ]:
# === Σύγκριση Static vs Temporal (για όλα τα μοντέλα) ===
print('\n' + '='*70)
print('BONUS: Static (T0 only) vs Temporal (T0+T1) Comparison')
print('='*70)

comparison_rows = []

models_to_compare = [
    ('Logistic Regression', lambda: LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42, solver='lbfgs', multi_class='multinomial')),
    ('Random Forest', lambda: RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)),
]

for name, model_fn in models_to_compare:
    print(f'\n--- {name} ---')
    
    # Static
    print('  [Static T0]')
    r_static = evaluate_model(model_fn, X_T0, y, model_name=f'{name} (Static)',
                              normalize=(name == 'Logistic Regression'))
    
    # Temporal
    print('  [Temporal T0+T1]')
    r_temporal = evaluate_model(model_fn, X_temporal, y, model_name=f'{name} (Temporal)',
                                normalize=(name == 'Logistic Regression'))
    
    comparison_rows.append({
        'Model': name,
        'Static Acc': f"{r_static['mean_acc']:.3f}±{r_static['std_acc']:.3f}",
        'Temporal Acc': f"{r_temporal['mean_acc']:.3f}±{r_temporal['std_acc']:.3f}",
        'Δ Acc': f"{r_temporal['mean_acc'] - r_static['mean_acc']:+.3f}",
    })

print('\n' + pd.DataFrame(comparison_rows).to_string(index=False))

---
## Συμπεράσματα

Τα αποτελέσματα θα χρησιμοποιηθούν για:
1. Ανανέωση Κεφαλαίου 3 (Μεθοδολογία)
2. Ανανέωση Κεφαλαίου 4 (Αποτελέσματα) 
3. Ανανέωση Κεφαλαίου 5 (Συζήτηση)